# 5.2.1 Model 2: Ridge Regression

Regularized Linear Regression model. Trained on `X_train_scaled.csv` (log1p-transformed numeric features, per Section 3.11.2) since Ridge Regression is scale-sensitive. L2 regularization (α = 1.0) was applied to reduce coefficient variance and improve generalization performance.

In [8]:
import pandas as pd
import numpy as np
import sys, os
from sklearn.linear_model import Ridge
import joblib
from model_utils import evaluate_model, cross_validate_model
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import RidgeCV
sys.path.append(os.path.dirname(os.path.abspath('__file__')))

MODELLING_DIR = os.path.join("..", "data", "modelling")

X_train = pd.read_csv(os.path.join(MODELLING_DIR, "X_train_scaled.csv"))
X_test = pd.read_csv(os.path.join(MODELLING_DIR, "X_test_scaled.csv"))
y_train = pd.read_csv(os.path.join(MODELLING_DIR, "y_train.csv"))["price"]
y_test = pd.read_csv(os.path.join(MODELLING_DIR, "y_test.csv"))["price"]

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")

X_train: (3004, 51) | X_test: (751, 51)


# Train Ridge Regression

We instantiate the Ridge model with `alpha=1.0` (the default) and fit it on the scaled training data. L2 regularisation shrinks the coefficients toward zero, which helps reduce overfitting and stabilises the model when features are correlated.

In [9]:
param_grid = {
    'alpha': np.logspace(-3, 3, 25)
}
ridge_cv = GridSearchCV(
    Ridge(),
    param_grid,
    scoring='neg_root_mean_squared_error',
    cv=5,
    n_jobs=-1
)
ridge_cv.fit(X_train, y_train)

print(f"Best alpha: {ridge_cv.best_params_['alpha']:.4f}")
ridge_model = ridge_cv.best_estimator_
print("Ridge model trained with tuned alpha.")

cv_scores = pd.DataFrame(ridge_cv.cv_results_)[['param_alpha', 'mean_test_score']]
cv_scores['mean_test_rmse'] = -cv_scores['mean_test_score']
print(cv_scores[['param_alpha', 'mean_test_rmse']].to_string(index=False))

Best alpha: 0.5623
Ridge model trained with tuned alpha.
 param_alpha  mean_test_rmse
    0.001000        0.277881
    0.001778        0.277881
    0.003162        0.277881
    0.005623        0.277881
    0.010000        0.277881
    0.017783        0.277880
    0.031623        0.277880
    0.056234        0.277879
    0.100000        0.277877
    0.177828        0.277874
    0.316228        0.277870
    0.562341        0.277866
    1.000000        0.277871
    1.778279        0.277910
    3.162278        0.278051
    5.623413        0.278446
   10.000000        0.279372
   17.782794        0.281260
   31.622777        0.284688
   56.234133        0.290393
  100.000000        0.299349
  177.827941        0.312708
  316.227766        0.331305
  562.341325        0.355132
 1000.000000        0.383249


# Evaluate

Metrics are computed on both the training and test sets. The gap between them is used later (Section 6.2) to assess overfitting/underfitting. All metrics are reported in **RM** (Ringgit Malaysia) after exponentiating the log‑transformed predictions, as defined in the project specification.

In [10]:
train_metrics = evaluate_model(
    ridge_model,
    X_train,
    y_train,
    label="Train"
)

print()

test_metrics = evaluate_model(
    ridge_model,
    X_test,
    y_test,
    label="Test"
)

Train RMSE:  RM 168,644  (48.2% of median price)
Train MAE:   RM 86,492
Train MAPE:  21.2%
Train R2:    0.7354
Train MSE:   28,440,684,164

Test RMSE:  RM 195,226  (54.2% of median price)
Test MAE:   RM 92,428
Test MAPE:  19.8%
Test R2:    0.6537
Test MSE:   38,113,136,675


# Coefficients

Ridge coefficients are shrunk compared to ordinary linear regression, but their signs should still broadly agree with the correlation directions observed in EDA – e.g. Property Size positive, Property Age negative. This serves as a basic sanity check for the model.

In [11]:
coef_table = pd.Series(
    ridge_model.coef_,
    index=X_train.columns
).sort_values(key=abs, ascending=False)

print("Top 15 coefficients by magnitude:")
print(coef_table.head(15))

Top 15 coefficients by magnitude:
Property Size                     0.837897
State_Perak                      -0.457645
Bedroom                          -0.425116
State_Negeri_Sembilan            -0.390051
PropertyType_Service_Residence    0.304394
Bathroom                          0.286113
State_Penang                      0.285153
State_Sabah                       0.281780
State_Melaka                     -0.269632
PropertyType_Flat                -0.237255
State_Pahang                      0.234939
State_Unknown                     0.215112
Parking Lot                       0.199352
State_Sarawak                     0.195990
State_Other                       0.156116
dtype: float64


# 5‑fold Cross‑Validation

Cross‑validation is run **only on the training set** (`X_train`) to obtain a more robust estimate of the model’s performance than a single train/test split. The test set remains untouched for the final evaluation. We use the same `alpha=1.0` without further tuning (a dedicated hyperparameter search would be a natural next step).

In [12]:
cv_results = cross_validate_model(
    Ridge(alpha=1.0),
    X_train,
    y_train,
    n_splits=5
)

5-fold CV (mean +/- std):
  RMSE:  RM 171,712 +/- 27,544  (48.8% of median price)
  MAE:   RM 88,637 +/- 5,260
  MAPE:  21.8% +/- 1.2%
  R2:    0.7200 +/- 0.0491
  MSE:   30,243,740,686 +/- 10,512,199,270


# Save Model

The fitted model is saved so the Streamlit prototype (Section 8) can load it directly without retraining.

In [13]:
import joblib

MODEL_DIR = os.path.join("..", "models")
os.makedirs(MODEL_DIR, exist_ok=True)

model_path = os.path.join(MODEL_DIR, "ridge_model.pkl")

joblib.dump(ridge_model, model_path)

print(f"Model saved to {model_path}")

Model saved to ..\models\ridge_model.pkl
